In [ ]:
from IPython.display import Markdown, display


In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path
from src.ingestion.pdf_parser import parse_pdf
from src.ingestion.chunker import load_and_chunk_pdf
from src.ingestion.embedder import embed_texts
from src.ingestion.ingest_pipeline import run_ingestion
from src.retrieval.retriever import retrieve
from src.retrieval.reranker import rerank
from src.retrieval.vector_store import VectorStore

To resolve the file path and load the pdf using env

In [ ]:
load_dotenv()
ROOT = Path(os.getenv("PYTHONPATH"))
# Resolve path from project root (notebook often runs with cwd = notebooks/)
ROOT = Path.cwd()
if not (ROOT / "uploaded_pdfs").exists():
    ROOT = ROOT.parent

To connecting the VectorDB

In [ ]:
vs = VectorStore()
print("ChromaDB connected")

Adding all the papers in the uploaded_pdfs. 

 **Dont run this multiple time it might cost you more because each time you add a duplicated it will delete the old one and replaces it with this one**

For Deleting all papers in CHROMA DB, **dont run this unless you need it**

In [ ]:
import shutil
from pathlib import Path

shutil.rmtree(ROOT / "chroma_db")
print("ChromaDB wiped")

To load all the datasets in to the DB.

In [ ]:
pdf_folder = ROOT / "uploaded_pdfs"

for pdf in pdf_folder.glob("*.pdf"):
    print(f"Ingesting: {pdf.name}")
    result = run_ingestion(str(pdf))
    print(f"Done — {result['num_chunks']} chunks")

In [ ]:
pdf_path = ROOT / "uploaded_pdfs" / "Attention_is _all_you_need.pdf"

# parse and chunk
chunks = load_and_chunk_pdf(str(pdf_path))
print(f"Total chunks: {len(chunks)}")

# embed
vectors = embed_texts(chunks)
print(f"Total vectors: {len(vectors)}")

# save to ChromaDB
vs.add(chunks, vectors, "attention")
print("Saved to ChromaDB")

This **.add** will add the pdf in VS by passsing the hash function i wrote later on in this project so we still can have duplicates if we add manually take care of that issue if you notice.

In [ ]:
#vs.add(chunks, vectors, "attention")
#print("Saved to ChromaDB")

In [ ]:
query = "What is the attention mechanism?"
query_vector = embed_texts([query])[0]

results = vs.search(query_vector, top_k=5)

for i, result in enumerate(results):
    print(f"Result {i+1}:")
    print(result["text"][:200])
    print(f"Score: {result['score']}")
    print("---")

We got 5 papers because the papers are distinguished with the paper name (Need to chnage this later in the project to remove duplicates automatically.)

I changed it and reduced the duplicates using hash.

** Just removed the cells where i removed those duplicates**

In [ ]:
from src.retrieval.vector_store import VectorStore
vs = VectorStore()
vs.list_papers()  # see the actual names

Testing wether the duplicate detcion is working or not.

In [ ]:
result = run_ingestion("uploaded_pdfs/Attention_is _all_you_need.pdf")
print(result)

The Duplicates removal has worked for now 

In [ ]:
results = retrieve("What is the attention mechanism?")

for i, result in enumerate(results[:5]):
    print(f"Result {i+1} (score: {result['score']:.4f}):")
    display(Markdown(result["text"]))
    print("---")

In [ ]:
results = retrieve("What is this project?")

for i, result in enumerate(results[:5]):
    print(f"Result {i+1} (score: {result['score']:.4f}):")
    display(Markdown(result["text"]))
    print("---")

In [ ]:
results = retrieve("What is paris located in the world?")

for i, result in enumerate(results[:5]):
    print(f"Result {i+1} (score: {result['score']:.4f}):")
    display(Markdown(result["text"]))
    print("---")

In [ ]:
results = retrieve("Does agentic retrieval works?")

for i, result in enumerate(results[:5]):
    print(f"Result {i+1} (score: {result['score']:.4f}):")
    display(Markdown(result["text"]))
    print("---")

In [ ]:
results = retrieve("what are the uses of RAG")

for i, result in enumerate(results[:5]):
    print(f"Result {i+1} (score: {result['score']:.4f}):")
    display(Markdown(result["text"]))
    print("---")

In [ ]:
results = retrieve("what is RAG")

for i, result in enumerate(results[:5]):
    print(f"Result {i+1} (score: {result['score']:.4f}):")
    display(Markdown(result["text"]))
    print("---")

In [ ]:
results = retrieve("How does retrieval-augmented generation combine retrieval with generation")

for i, result in enumerate(results[:5]):
    print(f"Result {i+1} (score: {result['score']:.4f}):")
    display(Markdown(result["text"]))
    print("---")

After implementing the RERANKER

In [ ]:
question = "How does retrieval-augmented generation combine retrieval with generation"
results = retrieve(question)
reranked = rerank(question, results, top_n=5)

for i, chunk in enumerate(reranked):
    print(f"Result {i+1} (score: {chunk['score']:.4f}):")
    display(Markdown(chunk["text"]))
    print("---")

In [ ]:
question = "What is the attention mechanism?"
results = retrieve(question)
reranked = rerank(question, results, top_n=5)

for i, chunk in enumerate(reranked):
    print(f"Result {i+1} (score: {chunk['score']:.4f}):")
    display(Markdown(chunk["text"]))
    print("---")